# Actividad individual 2 — Descubriendo patrones de tráfico con datos reales
## TC2035 — Diseño de Redes Neuronales y Aprendizaje Profundo

Nombre: Emilio Alonso Sánchez  
Matrícula: A00841853  
Grupo: 2  
Modalidad: Individual  
Herramienta: Google Colab (Python)  
Reto: Predicción inteligente de tiempos de verde — SEMEX

### Introducción

El objetivo de esta actividad es explorar los registros reales de la intersección 01 del Distrito Tec, Monterrey, para entender las diferencias de demanda antes de desarrollar una red neuronal para los tiempos de verde.

Se integran tres archivos Excel del estudio: mañana (07:00–10:00), mediodía (12:00–15:00) y tarde (17:00–20:00), cada uno con intervalos de 15 minutos. Para la comparación se conservan los movimientos 1, 2 y 3, identificados con los mismos números en las hojas de los tres archivos. También se analiza la hora de máxima demanda de toda la intersección.

### Estado de esta versión

El código, las tablas y las gráficas ya están adaptados a los tres periodos. Las celdas tituladas Interpretación 1–4 y Conclusión todavía corresponden a la versión anterior, que analizaba exclusivamente la mañana. Se conservan de forma provisional, por solicitud del estudiante; se actualizarán después de ejecutar y revisar los nuevos resultados. Esta versión todavía no está lista para entregarse.

No se entrena una MLP en esta actividad.

### Contexto y correspondencia de los archivos

Los tres libros tienen 16 hojas: una de hora máxima, 14 hojas de movimientos (1–14) y una hoja de volúmenes direccionales. El croquis utiliza una nomenclatura ligeramente diferente para los periodos:

| Archivo Excel | Nombre usado en el notebook | Periodo en el croquis | Intervalo del estudio | Hora máxima del croquis |
|---|---|---|---|---|
| Int. 01 mañana.xlsx | Mañana | Mañana | 07:00–10:00 | 08:00–09:00: 1,771 vehículos |
| Int. 01 mediodia.xlsx | Mediodía | Tarde | 12:00–15:00 | 13:45–14:45: 2,287 vehículos |
| Int. 01 tarde.xlsx | Tarde | Noche | 17:00–20:00 | 17:15–18:15: 2,372 vehículos |

Estos valores representan toda la intersección en su hora de máxima demanda, no solamente los tres movimientos seleccionados. El código calculará las ventanas móviles de 60 minutos a partir de los registros reales para cotejarlas con el croquis.

Los movimientos se identifican por número según sus hojas respectivas. No se atribuye un giro o dirección específica sin una leyenda que lo confirme.

---

## Parte 1. Exploración y comprensión de los datos

### 1.1. Carga de los tres archivos

El notebook busca los tres archivos en el directorio de trabajo; cuando falta alguno, intenta descargarlo desde la rama main del repositorio. En Google Colab, si la descarga falla, permite subirlo desde la computadora.

Los archivos son libros Excel (.xlsx), no CSV. Se utilizará Pandas para su lectura, manteniendo las filas originales con encabezados especiales antes de extraer los intervalos y los movimientos.

In [ ]:
# Importaciones y configuración: no se requiere TensorFlow para este análisis
from pathlib import Path
from io import BytesIO
from datetime import datetime, time
from urllib.parse import quote

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 25)
pd.set_option("display.precision", 4)

# Nombres exactos que aparecen en el repositorio
PERIODOS = [
    {"periodo": "Mañana", "archivo": "Int. 01 mañana.xlsx",
     "hora_inicio": 7, "hora_fin": 10, "pico_croquis": 1771,
     "inicio_pico_croquis": "08:00", "nombre_croquis": "Mañana"},
    {"periodo": "Mediodía", "archivo": "Int. 01 mediodia.xlsx",
     "hora_inicio": 12, "hora_fin": 15, "pico_croquis": 2287,
     "inicio_pico_croquis": "13:45", "nombre_croquis": "Tarde"},
    {"periodo": "Tarde", "archivo": "Int. 01 tarde.xlsx",
     "hora_inicio": 17, "hora_fin": 20, "pico_croquis": 2372,
     "inicio_pico_croquis": "17:15", "nombre_croquis": "Noche"}
]
ORDEN_PERIODOS = [p["periodo"] for p in PERIODOS]
BASE_REPO = "https://raw.githubusercontent.com/A00841853/Redes-Neuronales/main/"

def obtener_excel(nombre):
    if Path(nombre).is_file():
        print("Archivo local:", nombre)
        return str(Path(nombre))
    import requests
    try:
        respuesta = requests.get(BASE_REPO + quote(nombre), timeout=20)
        respuesta.raise_for_status()
        if respuesta.content[:2] != b"PK":
            raise ValueError("La respuesta no es un archivo XLSX válido.")
        print("Archivo descargado de GitHub:", nombre)
        return BytesIO(respuesta.content)
    except Exception as error:
        print("No se pudo descargar", nombre, ":", error)
        try:
            from google.colab import files
        except ImportError as exc:
            raise FileNotFoundError(
                f"Falta {nombre}. Descárgalo a la carpeta del notebook "
                "o ejecútalo en Colab para subirlo."
            ) from exc
        print("Sube el archivo:", nombre)
        subidos = files.upload()
        encontrado = next((k for k in subidos if k == nombre), None)
        if encontrado is None:
            raise FileNotFoundError(f"No se seleccionó el archivo esperado: {nombre}")
        return BytesIO(subidos[encontrado])

libros = {}
for p in PERIODOS:
    libros[p["periodo"]] = pd.ExcelFile(obtener_excel(p["archivo"]))

print("\nTres libros cargados:", ", ".join(libros.keys()))

In [ ]:
# Revisión de la estructura de los tres Excel
estructura = []
for periodo, libro in libros.items():
    hojas = libro.sheet_names
    estructura.append({
        "Periodo": periodo,
        "Hojas": len(hojas),
        "Movimientos numerados": sum(str(i) in hojas for i in range(1, 15)),
        "Contiene Hora Máx.": "Hora Máx." in hojas,
        "Contiene Vol. Dir.": "Vol. Dir." in hojas
    })
    print(periodo, "->", hojas)

tabla_estructura = pd.DataFrame(estructura)
display(tabla_estructura)

# Mostrar una vista original de las columnas usadas en cada periodo.
for p in PERIODOS:
    print(f"\nVista de movimiento 1 — {p['periodo']}:")
    original = pd.read_excel(libros[p["periodo"]], sheet_name="1", header=None)
    display(original.iloc[2:11, [0, 1, 4, 16]].fillna(""))

assert all(fila["Hojas"] == 16 and fila["Movimientos numerados"] == 14
           for fila in estructura)

### Interpretación 1. Comprendiendo los datos

¿Qué representa cada registro? Una fila válida corresponde a un intervalo de 15 minutos y al total de vehículos observados en un movimiento; no representa un vehículo individual.

¿Qué variables identifiqué? Los tiempos de inicio y fin del intervalo (columnas A y B), categorías vehiculares intermedias y el TOTAL del movimiento (columna Q). Cada movimiento aparece en una hoja identificada con su número, del 1 al 14.

¿Qué unidades tienen los volúmenes? Vehículos por intervalo de 15 minutos. Sumando cuatro registros consecutivos se obtiene el volumen de una hora.

¿Qué puede ser útil? El comportamiento por intervalo, los momentos de máxima demanda y las diferencias entre movimientos. También serían relevantes ocupación y colas si se recopilan en etapas posteriores.

¿Qué limpieza se requirió? El archivo tiene encabezados especiales y filas separadoras 14 y 19, que no son conteos. La lectura se realiza con `header=None` y se conservan solo filas con horas de inicio y fin identificables. En los movimientos 1, 2 y 3 se obtienen 12 intervalos completos, sin valores faltantes en Q. No se imputan datos inexistentes.

### 1.2. Extracción de los movimientos e intervalos

En los tres libros, cada movimiento aparece en su propia hoja numerada. Las columnas A y B registran el inicio y fin del intervalo, mientras que la columna Q almacena el volumen TOTAL del movimiento. Las columnas C:P contienen categorías de vehículos.

Se leen directamente las mismas hojas 1, 2 y 3 en cada periodo. Cuando una celda de Q carezca del resultado almacenado de su fórmula de Excel, el código reconstruirá ese total sumando exclusivamente las categorías vehiculares disponibles de esa misma fila. Si no hay ni total ni conteos de categorías, se notificará un error, en lugar de inventar datos.

El análisis reunirá 12 intervalos por periodo, para un total de 36 intervalos distintos.

In [ ]:
# Utilidades para extraer horas, totales y categorías de cada hoja
def leer_hora(valor):
    if pd.isna(valor):
        return None
    if isinstance(valor, (pd.Timestamp, datetime, time)):
        return f"{valor.hour:02d}:{valor.minute:02d}"
    if isinstance(valor, (int, float, np.number)) and not isinstance(valor, bool):
        x = float(valor)
        if 0 <= x < 1:
            minutos = round(x * 24 * 60) % (24 * 60)
            return f"{minutos // 60:02d}:{minutos % 60:02d}"
    if isinstance(valor, str):
        import re
        match = re.match(r"^\s*(\d{1,2}):(\d{2})", valor)
        if match:
            h, m = map(int, match.groups())
            if h < 24 and m < 60:
                return f"{h:02d}:{m:02d}"
    return None

def minutos_de_hora(h):
    return int(h[:2]) * 60 + int(h[3:5])

registro_reconstrucciones = []

def leer_movimiento(libro, periodo, numero, hora_inicio, hora_fin):
    """Usa A=inicio, B=fin, Q=total; reconstruye Q desde C:P si falta."""
    crudo = pd.read_excel(libro, sheet_name=str(numero), header=None)
    inicio = crudo.iloc[:, 0].map(leer_hora)
    fin = crudo.iloc[:, 1].map(leer_hora)
    hora_num = inicio.fillna("00:00").str[:2].astype(int)
    mascara = inicio.notna() & fin.notna() & hora_num.between(hora_inicio, hora_fin - 1)

    q = pd.to_numeric(crudo.iloc[:, 16], errors="coerce")
    categorias = crudo.iloc[:, 2:16].apply(pd.to_numeric, errors="coerce")
    suma_categorias = categorias.sum(axis=1, min_count=1)

    # Si Q no tiene el valor almacenado, se recupera de C:P sin crear datos.
    falta_q = mascara & q.isna()
    reconstruidos = int((falta_q & suma_categorias.notna()).sum())
    total = q.where(q.notna(), suma_categorias)

    # La presencia de una fila con hora y sin ningún conteo no se interpreta como cero.
    if total.loc[mascara].isna().any():
        raise ValueError(f"Hay intervalos sin dato de volumen en {periodo}, movimiento {numero}.")

    # Comprobar la coherencia del total cuando existen tanto Q como categorías.
    ambos = mascara & q.notna() & suma_categorias.notna()
    if (np.abs(q.loc[ambos] - suma_categorias.loc[ambos]) > 0.01).any():
        raise ValueError(f"El total Q no coincide con las categorías C:P en {periodo}, movimiento {numero}.")

    nombre = f"Movimiento {numero}"
    tabla = pd.DataFrame({
        "Intervalo": inicio.loc[mascara].to_numpy(),
        "Fin": fin.loc[mascara].to_numpy(),
        nombre: total.loc[mascara].to_numpy()
    })
    tabla[nombre] = pd.to_numeric(tabla[nombre])
    tabla = tabla.sort_values(
        "Intervalo", key=lambda c: c.map(minutos_de_hora)
    ).reset_index(drop=True)

    esperados = [f"{h:02d}:{m:02d}"
                 for h in range(hora_inicio, hora_fin)
                 for m in (0, 15, 30, 45)]
    if tabla["Intervalo"].tolist() != esperados:
        raise ValueError(f"Intervalos inesperados en {periodo}, movimiento {numero}.")
    registro_reconstrucciones.append({
        "Periodo": periodo, "Movimiento": numero,
        "Totales Q reconstruidos desde C:P": reconstruidos
    })
    return tabla

print("Funciones preparadas para leer mañana, mediodía y tarde.")

---

## Parte 2. Selección y visualización de movimientos vehiculares

Se seleccionan los movimientos 1, 2 y 3, que están presentes en los tres archivos. Se obtendrán 36 filas: 12 intervalos matutinos, 12 de mediodía y 12 de tarde. Al unirlos se preservará la etiqueta de periodo para evitar conectar horarios separados como si fueran observaciones continuas.

El análisis comparará tanto la demanda dentro de cada periodo como los volúmenes acumulados en las tres horas de cada levantamiento. Además, se contrastará la hora de máxima demanda de toda la intersección con los datos de la hoja Hora Máx. y las cifras del croquis.

In [ ]:
# Construir una tabla conjunta de los tres libros: 36 intervalos x 3 movimientos
movimientos = ["Movimiento 1", "Movimiento 2", "Movimiento 3"]
tablas = []

for info in PERIODOS:
    periodo = info["periodo"]
    libro = libros[periodo]
    series = [
        leer_movimiento(libro, periodo, numero,
                        info["hora_inicio"], info["hora_fin"])
        for numero in (1, 2, 3)
    ]
    subtabla = series[0]
    for serie in series[1:]:
        subtabla = subtabla.merge(
            serie, on=["Intervalo", "Fin"], how="inner", validate="one_to_one"
        )
    if len(subtabla) != 12:
        raise ValueError(f"Se esperaban 12 intervalos coincidentes en {periodo}.")
    subtabla.insert(0, "Periodo", periodo)
    subtabla["Minuto inicio"] = subtabla["Intervalo"].map(minutos_de_hora)
    subtabla["Suma de tres movimientos"] = subtabla[movimientos].sum(axis=1)
    tablas.append(subtabla)

datos = pd.concat(tablas, ignore_index=True)
datos["Periodo"] = pd.Categorical(
    datos["Periodo"], categories=ORDEN_PERIODOS, ordered=True
)

assert len(datos) == 36
assert datos.groupby("Periodo", observed=True).size().eq(12).all()
assert datos[movimientos].notna().all().all()

print("Tabla conjunta de los tres periodos:")
display(datos[["Periodo", "Intervalo", "Fin", *movimientos]])

print("\nReconstrucciones de totales Q (si eran necesarias):")
display(pd.DataFrame(registro_reconstrucciones))
print("Total de registros combinados:", len(datos))
print("Valores faltantes:", datos[movimientos].isna().sum().to_dict())

In [ ]:
# Tabla de acumulados por periodo y por movimiento
totales_periodo = datos.groupby("Periodo", observed=True)[movimientos].sum()
totales_periodo["Total movimientos 1–3"] = totales_periodo[movimientos].sum(axis=1)
display(Markdown("### Volúmenes acumulados en cada periodo de 3 horas"))
display(totales_periodo.astype(int))

# Comprobar los máximos horarios de toda la intersección con la hoja "Hora Máx."
# La hora pico puede comenzar a los 00, 15, 30 o 45 minutos.
registros_picos = []
intervalos_interseccion = {}

for info in PERIODOS:
    periodo = info["periodo"]
    tabla = pd.read_excel(libros[periodo], sheet_name="Hora Máx.", header=None)
    inicio = tabla.iloc[:, 0].map(leer_hora)
    fin = tabla.iloc[:, 1].map(leer_hora)
    horas = inicio.fillna("00:00").str[:2].astype(int)
    mascara = (
        inicio.notna() & fin.notna() &
        horas.between(info["hora_inicio"], info["hora_fin"] - 1)
    )

    total_inter = pd.to_numeric(tabla.iloc[:, 4], errors="coerce")
    base = pd.DataFrame({
        "Inicio": inicio.loc[mascara].to_numpy(),
        "Fin": fin.loc[mascara].to_numpy(),
        "Total intersección": total_inter.loc[mascara].to_numpy()
    })
    base = base.sort_values(
        "Inicio", key=lambda c: c.map(minutos_de_hora)
    ).reset_index(drop=True)

    if len(base) != 12 or base["Total intersección"].isna().any():
        raise ValueError(f"Datos incompletos en Hora Máx. para {periodo}.")

    # Ventanas móviles de cuatro intervalos de 15 minutos = 60 minutos
    movil = base["Total intersección"].rolling(4).sum()
    fin_ventana = int(movil.idxmax())
    principio_ventana = fin_ventana - 3
    hora_pico_inicio = base.loc[principio_ventana, "Inicio"]
    hora_pico_fin = base.loc[fin_ventana, "Fin"]
    volumen_pico = int(movil.iloc[fin_ventana])

    # Los tres movimientos se comparan en exactamente la misma hora.
    tres = datos.loc[datos["Periodo"] == periodo].reset_index(drop=True)
    volumen_tres = int(tres.iloc[principio_ventana:fin_ventana + 1][movimientos].sum().sum())

    registros_picos.append({
        "Periodo": periodo,
        "Hora pico inicio": hora_pico_inicio,
        "Hora pico fin": hora_pico_fin,
        "Intersección completa": volumen_pico,
        "Movimientos 1–3 en esa hora": volumen_tres,
        "Referencia croquis": info["pico_croquis"]
    })
    intervalos_interseccion[periodo] = base

    assert volumen_pico == info["pico_croquis"], \
        f"Volumen de hora pico no coincide con croquis en {periodo}."
    assert hora_pico_inicio == info["inicio_pico_croquis"], \
        f"Inicio de hora pico no coincide con croquis en {periodo}."

picos = pd.DataFrame(registros_picos).set_index("Periodo")
display(Markdown("### Hora máxima de cada periodo: Excel frente a croquis"))
display(picos)
print("Verificación satisfactoria: los tres valores de hora pico coinciden con el croquis.")

### 2.2. Relación con el croquis

El código calcula, para mañana, mediodía y tarde, las horas de máxima demanda a partir de la hoja Hora Máx. y verifica que coincidan con los totales publicados: 1,771, 2,287 y 2,372 vehículos. Esto evita confundir el volumen de toda la intersección con la suma de los movimientos 1, 2 y 3.

La suma de los movimientos seleccionados en cada hora pico se presenta por separado en la tabla de resultados. Las tres corrientes se mantienen identificadas con sus números originales. Para asignar un giro concreto o un sentido preciso a cada una sería necesaria la equivalencia del estudio.

In [ ]:
# Gráfica 1: evolución de los movimientos 1, 2 y 3, separados por periodo
# Los saltos entre 10:00, 12:00 y 17:00 NO se unen con líneas artificiales.
fig, ejes = plt.subplots(3, 1, figsize=(12, 12), sharey=True)
colores = ["#2563eb", "#059669", "#d97706"]

for ax, info in zip(ejes, PERIODOS):
    periodo = info["periodo"]
    sub = datos.loc[datos["Periodo"] == periodo]
    for nombre, color in zip(movimientos, colores):
        ax.plot(sub["Intervalo"], sub[nombre],
                marker="o", linewidth=2, markersize=5,
                label=nombre, color=color)
    ax.set_title(f"{periodo} | {info['hora_inicio']:02d}:00–{info['hora_fin']:02d}:00")
    ax.set_ylabel("Vehículos / 15 min")
    ax.grid(alpha=0.25)
    ax.legend(loc="upper left")
    ax.tick_params(axis="x", labelrotation=40)
    ax.set_xlabel("Inicio del intervalo")

fig.suptitle("Intersección 01: evolución de tres movimientos en los tres periodos",
             fontsize=14, y=1.015)
plt.tight_layout()
plt.show()

In [ ]:
# Gráfica 2: comparación conjunta de los volúmenes registrados en tres horas
# Cada barra muestra el TOTAL de un movimiento en un periodo de 3 horas.
fig, ax = plt.subplots(figsize=(10, 5.2))
posiciones = np.arange(len(ORDEN_PERIODOS))
ancho = 0.24

for j, (movimiento, color) in enumerate(zip(movimientos, colores)):
    valores = totales_periodo.loc[ORDEN_PERIODOS, movimiento].to_numpy()
    barras = ax.bar(posiciones + (j - 1) * ancho, valores,
                    width=ancho, label=movimiento, color=color)
    ax.bar_label(barras, padding=3, fontsize=9, fmt="%.0f")

ax.set_xticks(posiciones, ORDEN_PERIODOS)
ax.set_ylim(0, totales_periodo[movimientos].to_numpy().max() * 1.18)
ax.set_title("Volúmenes acumulados por movimiento y periodo")
ax.set_ylabel("Vehículos durante 3 horas")
ax.set_xlabel("Periodo del estudio")
ax.grid(axis="y", alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

print("Resumen de los tres movimientos por periodo:")
display(totales_periodo.astype(int))
print("Movimiento de mayor volumen en cada periodo:")
for periodo in ORDEN_PERIODOS:
    ganador = totales_periodo.loc[periodo, movimientos].idxmax()
    print(f"- {periodo}: {ganador}")

### Interpretación 2. Comportamiento del tráfico

¿Cuál de los tres movimientos presenta mayor volumen? El Movimiento 1, con 842 vehículos entre las 07:00 y las 10:00, seguido del Movimiento 2 (491) y el Movimiento 3 (188). De los 1,521 vehículos acumulados entre los tres movimientos durante esas tres horas, representan aproximadamente 55.4 %, 32.3 % y 12.4 %, respectivamente.

¿En qué intervalo hay mayor demanda? Es importante distinguir dos escalas temporales:

| Ventana de comparación | Resultado de los tres movimientos |
|---|---|
| 15 minutos | 09:00–09:15: 164 vehículos (75 + 72 + 17) |
| Una hora completa | 08:00–09:00: 548 vehículos (291 + 179 + 78) |

Al sumar bloques horarios consecutivos, los tres movimientos registran 446 vehículos de 07:00 a 08:00, 548 de 08:00 a 09:00 y 527 de 09:00 a 10:00. Por tanto, aunque el intervalo individual más cargado comienza a las 09:00, la hora completa con mayor suma es 08:00–09:00. El croquis registra para esa misma hora 1,771 vehículos en toda la intersección; este total general incluye movimientos adicionales.

¿Se comportan de manera similar? No exactamente. El Movimiento 1 alcanza su máximo de 89 vehículos entre 08:30 y 08:45; el Movimiento 2, 72 entre 09:00 y 09:15; y el Movimiento 3, 28 entre 07:45 y 08:00. La gráfica de líneas confirma que sus picos no coinciden.

¿Qué diferencias se observan? El Movimiento 1 mantiene un volumen generalmente mayor, mientras que el Movimiento 3 registra conteos más pequeños. El Movimiento 2 presenta un pico destacado a las 09:00, a diferencia del Movimiento 3, cuyo mayor registro ocurre antes de las 08:00. Sus fluctuaciones no son idénticas, por lo que comparar únicamente los totales ocultaría diferencias importantes entre intervalos.

¿Por qué es relevante para los tiempos de verde? Estas diferencias podrían orientar decisiones sobre cómo atender la demanda de las distintas corrientes. Sin embargo, los conteos por sí solos no determinan los segundos de verde recomendados: también deben considerarse capacidad de carriles, colas, fases compatibles, tiempos peatonales y restricciones de seguridad.

In [ ]:
# Resúmenes complementarios para interpretar las gráficas sin confundir escalas
detalle_maximos = []
for info in PERIODOS:
    periodo = info["periodo"]
    sub = datos.loc[datos["Periodo"] == periodo].copy()
    fila = sub.loc[sub["Suma de tres movimientos"].idxmax()]
    detalle_maximos.append({
        "Periodo": periodo,
        "Mayor intervalo 15 min": f"{fila['Intervalo']}–{fila['Fin']}",
        "Vehículos en 15 min (mov. 1–3)": int(fila["Suma de tres movimientos"]),
        "Movimiento mayor en 3 h":
            totales_periodo.loc[periodo, movimientos].idxmax()
    })
display(Markdown("### Intervalos de mayor demanda entre los tres movimientos"))
display(pd.DataFrame(detalle_maximos).set_index("Periodo"))

# Gráfica 3: proporción de los tres movimientos dentro de la hora pico GENERAL
fig, ax = plt.subplots(figsize=(10, 5.2))
xp = np.arange(len(ORDEN_PERIODOS))
todo = picos.loc[ORDEN_PERIODOS, "Intersección completa"].to_numpy()
parte = picos.loc[ORDEN_PERIODOS, "Movimientos 1–3 en esa hora"].to_numpy()

barras1 = ax.bar(xp - 0.18, todo, width=0.36, label="Intersección completa",
                 color="#64748b")
barras2 = ax.bar(xp + 0.18, parte, width=0.36, label="Movimientos 1–3",
                 color="#2563eb")
ax.bar_label(barras1, padding=3, fmt="%d")
ax.bar_label(barras2, padding=3, fmt="%d")
ax.set_xticks(xp, ORDEN_PERIODOS)
ax.set_ylabel("Vehículos en la hora de máxima demanda")
ax.set_title("Demanda de toda la intersección frente a los tres movimientos")
ax.set_ylim(0, max(todo) * 1.17)
ax.grid(axis="y", alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

print("Las barras son conteos de UNA HORA, no sumas de tres horas.")
print("Los tres totales generales corresponden a ventanas móviles de 60 min.")

---

## Parte 3. Normalización conjunta de los datos

Para comparar los tres periodos, la función Min-Max se ajusta una sola vez para cada movimiento utilizando conjuntamente sus 36 registros, sin reiniciar el mínimo y el máximo al cambiar de periodo:

\[
x_{\text{normalizado}}=\frac{x-x_{\min}}{x_{\max}-x_{\min}}.
\]

Así se conservan las diferencias relativas de nivel entre mañana, mediodía y tarde, además de las fluctuaciones dentro de cada periodo. Si se normalizara cada periodo por separado, los tres periodos mostrarían máximos de 1 incluso cuando sus volúmenes reales fueran diferentes, dificultando una comparación entre horarios.

Cuando una serie es constante se le asigna 0 a sus valores válidos para evitar dividir entre cero, dejando constancia de ello.

Esta transformación es exclusivamente didáctica. Para una MLP real, el escalador debe ajustarse solo con datos de entrenamiento y reutilizarse para validación y prueba, sin fuga de información.

In [ ]:
# Normalización global por movimiento (36 registros combinados).
# Se conservan los valores originales y el periodo de cada observación.
normalizados = pd.DataFrame(index=datos.index)
minmax = []
constantes = []

for movimiento in movimientos:
    serie = datos[movimiento].astype(float)
    minimo, maximo = serie.min(), serie.max()
    rango = maximo - minimo

    if np.isclose(rango, 0):
        normalizados[movimiento] = serie.apply(
            lambda x: np.nan if pd.isna(x) else 0.0
        )
        constantes.append(movimiento)
    else:
        normalizados[movimiento] = (serie - minimo) / rango

    minmax.append({
        "Movimiento": movimiento,
        "Mínimo conjunto": minimo,
        "Máximo conjunto": maximo
    })

display(Markdown("### Extremos de Min-Max calculados con los tres periodos"))
display(pd.DataFrame(minmax))

tabla_normalizacion = datos[["Periodo", "Intervalo"]].copy()
for movimiento in movimientos:
    tabla_normalizacion[f"{movimiento} original"] = datos[movimiento]
    tabla_normalizacion[f"{movimiento} Min-Max"] = normalizados[movimiento]

display(Markdown("### Valores originales y transformados: 36 registros"))
display(tabla_normalizacion.round(4))
print("Movimientos constantes:", constantes or "ninguno")

valores = normalizados.to_numpy(dtype=float)
finitos = valores[np.isfinite(valores)]
assert len(finitos) > 0
assert (finitos >= -1e-10).all()
assert (finitos <= 1 + 1e-10).all()

In [ ]:
# Gráfica 4: comparación de Min-Max sin reajustar la escala en cada periodo
fig, ejes = plt.subplots(3, 1, figsize=(12, 12), sharey=True)

for ax, info in zip(ejes, PERIODOS):
    periodo = info["periodo"]
    mascara = datos["Periodo"] == periodo
    sub = datos.loc[mascara]
    for movimiento, color in zip(movimientos, colores):
        ax.plot(sub["Intervalo"], normalizados.loc[mascara, movimiento],
                marker="o", linewidth=2, markersize=5,
                label=movimiento, color=color)
    ax.set_title(f"{periodo} | Escala común para todos los periodos")
    ax.set_ylabel("Min-Max conjunto (0–1)")
    ax.set_ylim(-0.06, 1.07)
    ax.set_xlabel("Inicio del intervalo")
    ax.grid(alpha=0.25)
    ax.tick_params(axis="x", labelrotation=40)
    ax.legend(loc="upper left")

fig.suptitle("Volúmenes normalizados de los movimientos 1–3 en tres periodos",
             fontsize=14, y=1.015)
plt.tight_layout()
plt.show()

### Interpretación 3. Normalización

¿Qué cambió? Las tres series dejaron de expresarse en vehículos por 15 minutos y pasaron a una escala adimensional de 0 a 1, calculada por separado para cada movimiento. Así se facilita la comparación de las fluctuaciones relativas, aunque los volúmenes originales siguen siendo importantes para interpretar la demanda real.

¿Se conservaron los patrones de crecimiento y disminución? Sí. Como Min-Max es una transformación lineal creciente para las series que varían, se mantienen los máximos y mínimos en los mismos intervalos. La gráfica normalizada conserva los máximos de 08:30 en el Movimiento 1, 09:00 en el 2 y 07:45 en el 3.

¿Por qué puede ayudar a entrenar una MLP? Las entradas en rangos comparables pueden facilitar la optimización y evitar que una variable tenga mayor influencia inicial únicamente por su magnitud numérica. El escalado no crea información ni elimina la necesidad de contar con suficientes observaciones.

¿Significa que todos los movimientos tienen el mismo volumen? No. Un valor normalizado de 1 representa 89 vehículos en el Movimiento 1, 72 en el 2 y 28 en el 3. Asimismo, un valor normalizado de 0 no significa ausencia de tráfico: a las 07:00 corresponde a 39, 20 y 8 vehículos, respectivamente. Significa que cada movimiento se encuentra en su mínimo observado en los datos utilizados.

¿Qué ocurre con una serie constante? Si \(x_{\max}=x_{\min}\), se evita dividir entre cero asignando 0 a los valores válidos de esa serie y documentando que no hay variación.

Precaución: para este análisis exploratorio se usan los 12 intervalos. En una MLP real, los valores mínimo y máximo deben estimarse solo con el conjunto de entrenamiento y aplicarse después sin reajustarlos a validación, prueba y producción.

---

## Parte 4. Identificación de patrones para una futura MLP

### Propuesta de variables de entrada

| Variable candidata | ¿Qué información aporta? |
|---|---|
| Volumen del Movimiento 1 (veh./15 min) | Demanda de la primera corriente del acceso estudiado. |
| Volumen del Movimiento 2 (veh./15 min) | Demanda de la segunda corriente, distinta de la primera. |
| Volumen del Movimiento 3 (veh./15 min) | Demanda de la tercera corriente, de menor volumen en este estudio. |
| Hora del día | Identificación de patrones temporales de llegadas. |
| Ocupación y longitud de cola (requieren otros datos) | Congestión y acumulación de vehículos antes del semáforo. |

La MLP podría aprender relaciones entre hora, flujos de distintos movimientos, ocupación y colas. Los patrones detectados sugieren que distintos movimientos alcanzan su mayor demanda en momentos diferentes, y el controlador podría necesitar adaptar sus decisiones.

Entradas frente a etiquetas: los conteos de tráfico serían entradas. Las etiquetas objetivo serían tiempos de verde adecuados por fase (en segundos), obtenidos de registros confiables o mediante un esquema de optimización y validación. El Excel de volúmenes no proporciona por sí solo tales etiquetas, por lo que no es apropiado fabricar objetivos de entrenamiento. También se requieren reglas de fases, tiempos mínimos, intervalos de seguridad, demanda peatonal y restricciones del controlador.

### Interpretación 4. Conexión con las redes neuronales

¿Qué datos utilizaría como entradas de una MLP? Los volúmenes de los movimientos 1, 2 y 3 por intervalo, junto con la hora. Para un sistema más completo también incorporaría otros accesos, ocupación, longitudes de cola, velocidades y demanda peatonal.

¿Qué me gustaría predecir? Tiempos de verde o *splits* recomendados por fase que respondan a la demanda y cumplan con las reglas de operación segura de la intersección.

¿Disponemos actualmente de las etiquetas necesarias? No se identificaron tiempos de verde recomendados como variables objetivo en este archivo de aforos. Conocer que el Movimiento 1 registra más vehículos no permite determinar automáticamente cuántos segundos de verde debería recibir.

¿Qué diferencia existe entre ver un patrón y demostrar que una red lo aprendió? Las gráficas describen estos 12 intervalos de una sola mañana, pero no prueban que una MLP pueda generalizar a otras fechas o condiciones. Para demostrar aprendizaje se necesita entrenar con pares entrada–objetivo confiables y evaluar predicciones no vistas.

¿Cómo comprobaría después que funciona? Reuniría datos de más días y periodos, establecería separaciones temporales entre entrenamiento y prueba, ajustaría el escalador únicamente con entrenamiento y evaluaría MAE/RMSE frente a referencias de verde válidas. Posteriormente compararía políticas mediante simulación calibrada, usando demoras, colas, tiempos de espera por movimiento y verificaciones de seguridad. No basta con predecir bien las etiquetas: hay que comprobar la utilidad operativa.

### Verificación del análisis conjunto

El siguiente bloque verifica que se leyeron los tres libros Excel, cada uno con 16 hojas, y que el conjunto contiene 36 intervalos válidos (12 por periodo) para los movimientos 1, 2 y 3. También confirma que las horas máximas de toda la intersección coinciden con el croquis.

Las cuatro interpretaciones y la conclusión escritas más arriba siguen siendo las de la versión matutina anterior. Quedan pendientes de ajustar después de ejecutar este nuevo código, revisar todas las tablas y gráficos y confirmar los nuevos patrones.

In [ ]:
# Validación final reproducible para los tres periodos combinados
hojas_esperadas = ["Hora Máx.", *[str(i) for i in range(1, 15)], "Vol. Dir."]
for periodo, libro in libros.items():
    assert libro.sheet_names == hojas_esperadas, f"Hojas inesperadas en {periodo}"

assert len(datos) == 36
assert (datos.groupby("Periodo", observed=True).size() == 12).all()
assert not datos[movimientos].isna().any().any()
assert len(registro_reconstrucciones) == 9  # 3 movimientos x 3 periodos
assert normalizados.shape == (36, 3)

assert picos.loc[ORDEN_PERIODOS, "Intersección completa"].astype(int).tolist() == [
    1771, 2287, 2372
]
assert picos.loc[ORDEN_PERIODOS, "Hora pico inicio"].tolist() == [
    "08:00", "13:45", "17:15"
]

print("Validación satisfactoria de tres Excel y tres periodos.")
print("Total:", len(datos), "intervalos con tres movimientos por intervalo.")
print("Se generaron tablas comparativas y gráficas de volúmenes originales y normalizados.")
print("Los tres máximos horarios coinciden con el croquis.")
print("Todavía no se entrenó una red neuronal.")
print("PENDIENTE: actualizar interpretaciones y conclusión con los resultados de esta versión.")

---

## Conclusión: ¿qué aprendí sobre los datos de tráfico?

Esta actividad me permitió comprender la importancia de analizar los datos antes de construir una red neuronal. Al revisar el Excel de la intersección 01, identifiqué 12 registros de 15 minutos para cada movimiento seleccionado y observé que las condiciones de tráfico cambian a lo largo de la mañana.

Entre las 07:00 y las 10:00, el Movimiento 1 acumuló 842 vehículos, el Movimiento 2 registró 491 y el Movimiento 3 tuvo 188. Sus máximos individuales ocurrieron a distintas horas, por lo que no siguen exactamente el mismo comportamiento. También aprendí a distinguir entre analizar un intervalo y una hora completa: aunque la suma más alta en 15 minutos fue de 164 vehículos entre 09:00 y 09:15, la hora con mayor volumen acumulado de los tres movimientos fue de 08:00 a 09:00, con 548 vehículos.

La normalización Min-Max me ayudó a comparar las variaciones relativas de cada movimiento sin confundirlas con sus volúmenes absolutos. Por ejemplo, un valor normalizado de cero no significa necesariamente que no circularon vehículos.

Finalmente, entendí que estos datos pueden ser entradas de una futura MLP para el reto de SEMEX, pero aún faltan observaciones de otros días, información sobre colas, etiquetas de tiempos de verde y restricciones operativas. Antes de afirmar que un modelo mejora el tránsito, sería indispensable comprobar que sus predicciones son seguras y realmente reducen las demoras.